In [4]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download(
    "mohamedbakhet/amazon-books-reviews"
)

ratings_path = path + "/Books_rating.csv"

df = pd.read_csv(
    ratings_path,
    usecols=["User_id", "Title", "review/score"],
    nrows=300000
)

In [5]:
df = df.dropna()

df = df[df["review/score"] >= 4]

book_stats = df.groupby("Title").agg(
    avg_rating=("review/score", "mean"),
    rating_count=("review/score", "count")
).reset_index()

book_stats = book_stats[book_stats["rating_count"] >= 10]

In [6]:
def recommend_books(user_id, top_n=10):
    user_books = df[df["User_id"] == user_id]["Title"].unique()

    if len(user_books) == 0:
        return book_stats.sort_values(
            ["avg_rating", "rating_count"],
            ascending=False
        ).head(top_n)

    similar_users = df[df["Title"].isin(user_books)]["User_id"].unique()

    recommendations = df[
        (df["User_id"].isin(similar_users)) &
        (~df["Title"].isin(user_books))
    ]

    if recommendations.empty:
        return book_stats.sort_values(
            ["avg_rating", "rating_count"],
            ascending=False
        ).head(top_n)

    result = recommendations.groupby("Title").agg(
        avg_rating=("review/score", "mean"),
        rating_count=("review/score", "count")
    ).reset_index()

    result = result.sort_values(
        ["avg_rating", "rating_count"],
        ascending=False
    )

    return result.head(top_n)


print(df["User_id"].head())

active_user = df["User_id"].value_counts().index[0]

print("User:", active_user)

print(recommend_books(active_user, top_n=10))

0     AVCGYZL8FQQTD
1    A30TK6U7DNS82R
2    A3UH4UZ4RSVO82
3    A2MVUWT453QH61
4    A22X4XUPKF66MR
Name: User_id, dtype: str
User: A14OJS0VWMOSWO
                                                  Title  avg_rating  \
840                                      Complete Works         5.0   
2994                   Shadow of The Moon (Part 1 of 2)         5.0   
3521                  The Complete Works of Shakespeare         5.0   
4365                  The life of Samuel Johnson, LL. D         5.0   
2864                           Rubaiyat of Omar Khayyam         5.0   
2934  Scientific Progress Goes 'Boink': A Calvin and...         5.0   
3631                                 The Fire Next Time         5.0   
3816              The Letter of Marque (Aubrey-Maturin)         5.0   
3513  The Complete Peanuts, 1957- 1958 (The Complete...         5.0   
401   Arctic Dreams: Imagination and Desire in a Nor...         5.0   

      rating_count  
840             10  
2994            10  
3521    